# TRELLIS.2 – Text/Image to 3D (Flash Attention 2)
دفتر ملاحظات لتشغيل مشروع [TRELLIS.2-Text-to-3D-FA2](https://github.com/PRITHIVSAKTHIUR/TRELLIS.2-Text-to-3D-FA2) على Google Colab.

**ملاحظة هامة:** Flash Attention 2 يتطلب معالج رسومي من فئة Ampere أو أحدث (A100, A6000, RTX 3090/4090). في بيئة Colab المجانية (T4) سيتم استخدام الانتباه العادي (SDPA) تلقائياً.
للحصول على أقصى أداء يُنصح باستخدام مسرّع **A100** (Colab Pro).

In [ ]:
import torch
import subprocess, sys

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if not torch.cuda.is_available():
    raise RuntimeError("الرجاء تفعيل مسرّع GPU من القائمة: runtime → Change runtime type → GPU")

!nvidia-smi
gpu_name = subprocess.getoutput("nvidia-smi --query-gpu=name --format=csv,noheader")
print(f"GPU: {gpu_name}")

# Flash Attention 2 يحتاج compute capability >= 8.0 (Ampere)
install_flash = any(x in gpu_name for x in ["A100", "A6000", "3090", "4090", "L4", "L40"])
if install_flash:
    print("✅ GPU يدعم Flash Attention 2")
else:
    print("ℹ️ GPU لا يدعم Flash Attention، سيتم استخدام SDPA (انتباه PyTorch العادي)")

In [ ]:
import os, sys

# استنساخ المستودع
if not os.path.exists("TRELLIS.2-Text-to-3D-FA2"):
    !git clone https://github.com/PRITHIVSAKTHIUR/TRELLIS.2-Text-to-3D-FA2.git
%cd TRELLIS.2-Text-to-3D-FA2

# تثبيت spconv (مطلوب لـ TRELLIS)
cuda_ver = torch.version.cuda.replace(".", "")
!pip install spconv-cu{cuda_ver} 2>&1 | tail -5

# تثبيت المتطلبات الأخرى
!pip install -r requirements.txt 2>/dev/null || echo "No requirements.txt"
!pip install trimesh pygltflib viser omegaconf opencv-python imageio[ffmpeg] gradio pyngrok 2>&1 | tail -5

# تثبيت flash-attn إذا كان مدعوماً
if install_flash:
    !pip install ninja packaging
    !pip install flash-attn --no-build-isolation 2>&1 | tail -5
    print("✅ Flash Attention 2 تم تثبيته")
else:
    print("⏩ تخطي تثبيت flash-attn")

# تثبيت الحزمة من الكود المصدري
!pip install -e . 2>&1 | tail -10
print("✅ تم تثبيت TRELLIS.2")

In [ ]:
import torch
from trellis.pipelines import TrellisImageTo3DPipeline

# تحميل خط الأنابيب (يستخدم Flash Attention تلقائياً إن وُجد)
pipeline = TrellisImageTo3DPipeline.from_pretrained(
    "JeffreyXiang/TRELLIS-image-large",
    torch_dtype=torch.float16
)
pipeline.to("cuda")
print("✅ تم تحميل خط الأنابيب")

In [ ]:
from PIL import Image
import requests

# تحميل صورة تجريبية (أو ارفع صورتك الخاصة)
url = "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/diffusers/tiger.png"
image = Image.open(requests.get(url, stream=True).raw).convert("RGB")
display(image)

In [ ]:
# توليد المجسم الثلاثي الأبعاد
outputs = pipeline.run(
    image,
    seed=42,
    # يمكن تعديل البارامترات مثل steps, cfg, إلخ.
)
print("🔹 تم الانتهاء من التوليد")
print("المفاتيح الناتجة:", outputs.keys())
# المخرجات تحتوي على 'gaussian' (سحابة نقاط غاوسية) و 'mesh' (شبكة مثلثية)

In [ ]:
import numpy as np
from pathlib import Path

out_dir = Path("outputs")
out_dir.mkdir(exist_ok=True)

# تصدير الشبكة (mesh) بصيغة GLB إذا كانت متاحة
mesh = outputs.get('mesh')
if mesh is not None:
    glb_path = out_dir / "model.glb"
    mesh.export(glb_path)
    print(f"✅ تم تصدير الشبكة إلى {glb_path}")
else:
    print("⚠️ لا توجد شبكة في المخرجات")

# تصدير السحابة الغاوسية بصيغة PLY (إن احتجتها)
gaussian = outputs.get('gaussian')
if gaussian:
    ply_path = out_dir / "gaussian.ply"
    # دالة save_ply قد لا تكون موجودة في كل الإصدارات، نتحقق
    if hasattr(gaussian[0], 'save_ply'):
        gaussian[0].save_ply(ply_path)
        print(f"✅ تم تصدير Gaussians إلى {ply_path}")
    else:
        print("⚠️ لا توجد دالة save_ply، يمكن استخدام بيانات xyz مباشرة")

In [ ]:
# إعداد عرض تفاعلي ثلاثي الأبعاد باستخدام viser + ngrok
import viser
from trellis.utils import render_utils
from pyngrok import ngrok
import time

# إنهاء أي أنفاق قديمة
ngrok.kill()

# تشغيل خادم viser على المنفذ 8080
server = viser.ViserServer(host="0.0.0.0", port=8080)

# إضافة السحابة الغاوسية إلى المشهد
if gaussian:
    render_utils.render_gaussian(server, gaussian[0])
    print("✅ تمت إضافة المجسم إلى العارض")

# فتح نفق ngrok
public_url = ngrok.connect(8080).public_url
print(f"🔗 رابط المشاهدة التفاعلية: {public_url}")
print("⚠️ قد تحتاج إلى الانتظار بضع ثوانٍ ثم فتح الرابط في المتصفح")

In [ ]:
from IPython.display import IFrame, display

# عرض العارض داخل دفتر الملاحظات (قد لا يعمل في كل المتصفحات)
display(IFrame(src=public_url, width="100%", height="600px"))

## ملاحظات
- إذا كنت تستخدم GPU من نوع T4 (Colab مجاني)، سيتم تجاهل Flash Attention ويعمل النموذج بشكل طبيعي ولكن بسرعة أقل.
- يمكنك رفع صورتك الخاصة بدلاً من الرابط التجريبي.
- الملفات المُصدَّرة `model.glb` و `gaussian.ply` ستظهر في مجلد `outputs` داخل بيئة Colab ويمكن تحميلها.